# 07 TradeSkip Threshold Research

Исследуем рабочие режимы уже обученного TradeSkip: хотим сохранить качество, но получить больше сделок.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.features.event_detector import detect_events
from src.features.feature_pipeline import generate_features
from src.models.sequence_models import load_model_with_config
from src.models.trade_skip_training import generate_trade_skip_signal_history
from src.strategy.backtest import build_trades, calculate_trade_metrics
from src.strategy.signal_generator import generate_rule_based_signal_history

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

## Настройки

Модели не переобучаем. Только подбираем threshold и проверяем устойчивость на разных окнах.

In [2]:
MODEL_TYPES = ["gru", "lstm"]

HORIZON = config.DEFAULT_HORIZON_CANDLES
TP_THRESHOLD = config.DEFAULT_TP_THRESHOLD
SL_THRESHOLD = config.DEFAULT_SL_THRESHOLD

THRESHOLD_GRID = np.round(np.arange(0.40, 0.61, 0.01), 2)
Q_CANDLES_LIST = [2000, 5000, 10000]

VALID_START = pd.Timestamp(config.TRAIN_END_DATE)
TEST_START = pd.Timestamp(config.VALID_END_DATE)

MIN_TRADES_BY_MODE = {
    "quality": 15,
    "balanced": 50,
    "active": 100,
}

In [3]:
# Данные и подготовленный фрейм.
price_df, loaded_files = load_all_price_data(PROJECT_ROOT / "data")
prepared_df = detect_events(generate_features(price_df))

print(f"CSV файлов: {len(loaded_files)}")
print(f"Свечей: {len(price_df):,}")
print(f"Events: {int(prepared_df['event'].sum()):,}")
print(f"Период: {prepared_df.index.min()} -> {prepared_df.index.max()}")

CSV файлов: 45
Свечей: 277,105
Events: 21,684
Период: 2015-01-02 17:45:00+00:00 -> 2026-03-26 16:30:00+00:00


In [4]:
def trade_skip_paths(model_type: str):
    model_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_best.pth"
    scaler_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_scaler.pkl"
    config_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_config.pkl"
    return model_path, scaler_path, config_path


def localize_like_index(ts: pd.Timestamp, index: pd.Index) -> pd.Timestamp:
    if getattr(index, "tz", None) is not None and ts.tzinfo is None:
        return ts.tz_localize(index.tz)
    return ts


VALID_START_TS = localize_like_index(VALID_START, prepared_df.index)
TEST_START_TS = localize_like_index(TEST_START, prepared_df.index)


def select_period(signals: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp | None = None, q_candles: int | None = None) -> pd.DataFrame:
    result = signals[signals["time"].ge(start)].copy()
    if end is not None:
        result = result[result["time"].lt(end)].copy()
    if q_candles is not None:
        result = result.tail(q_candles).copy()
    return result


def apply_trade_skip_threshold(signals: pd.DataFrame, threshold: float) -> pd.DataFrame:
    result = signals.copy()
    can_trade = result["event"].eq(1) & result["probability_trade"].ge(threshold)
    result["decision"] = "NO TRADE"
    result.loc[can_trade & result["event_cusum_direction"].eq(1), "decision"] = "BUY"
    result.loc[can_trade & result["event_cusum_direction"].eq(-1), "decision"] = "SELL"
    return result


def summarize_trades(name: str, signals: pd.DataFrame, threshold: float, q_candles: int, period: str) -> dict:
    filtered = apply_trade_skip_threshold(signals, threshold) if "probability_trade" in signals.columns else signals.copy()
    trades = build_trades(filtered, prepared_df, horizon=HORIZON, tp_threshold=TP_THRESHOLD, sl_threshold=SL_THRESHOLD)
    metrics = calculate_trade_metrics(trades)
    return {
        "period": period,
        "q_candles": q_candles,
        "strategy": name,
        "threshold": threshold,
        "trades": metrics["Trades"],
        "total_return": metrics["Total Return"],
        "winrate": metrics["Win Rate"],
        "profit_factor": metrics["Profit Factor"],
        "max_drawdown": metrics["Max Drawdown"],
        "avg_trade": metrics["Average Trade"],
    }


def score_mode(row: pd.Series, min_trades: int) -> float:
    # Доходность важна, но слишком редкие и убыточные режимы штрафуем.
    score = row["total_return"]
    if row["trades"] < min_trades:
        score -= 10.0
    if row["profit_factor"] < 1.0:
        score -= 1.0
    if row["max_drawdown"] < -0.05:
        score += row["max_drawdown"]
    return score

## Сырые вероятности

Генерируем probability_trade один раз, дальше быстро гоняем thresholds.

In [5]:
raw_by_model = {}
for model_type in MODEL_TYPES:
    model_path, scaler_path, config_path = trade_skip_paths(model_type)
    model_config = joblib.load(config_path)
    model = load_model_with_config(model_path, config_path)
    scaler = joblib.load(scaler_path)
    raw = generate_trade_skip_signal_history(
        price_df,
        model=model,
        scaler=scaler,
        feature_columns=model_config["feature_columns"],
        threshold=0.0,
        max_rows=len(prepared_df),
    )
    raw_by_model[model_type] = raw
    print(model_type, len(raw), raw["probability_trade"].describe())

gru 276827 count    276827.000000
mean          0.496192
std           0.032775
min           0.359889
25%           0.476601
50%           0.506177
75%           0.522909
max           0.560112
Name: probability_trade, dtype: float64
lstm 276827 count    276827.000000
mean          0.505381
std           0.030043
min           0.244421
25%           0.490880
50%           0.506539
75%           0.525570
max           0.569161
Name: probability_trade, dtype: float64


## Validation sweep

Threshold выбираем на validation. Смотрим три режима: quality, balanced, active.

In [6]:
valid_rows = []
for q_candles in Q_CANDLES_LIST:
    for model_type, raw in raw_by_model.items():
        valid_signals = select_period(raw, VALID_START_TS, TEST_START_TS, q_candles=q_candles)
        for threshold in THRESHOLD_GRID:
            valid_rows.append(summarize_trades(f"TradeSkip + {model_type.upper()}", valid_signals, float(threshold), q_candles, "valid"))

valid_df = pd.DataFrame(valid_rows)

selected_rows = []
for mode, min_trades in MIN_TRADES_BY_MODE.items():
    scored = valid_df.copy()
    scored["mode"] = mode
    scored["min_trades"] = min_trades
    scored["selection_score"] = scored.apply(lambda row: score_mode(row, min_trades), axis=1)
    selected_rows.append(
        scored.sort_values(["selection_score", "profit_factor", "total_return"], ascending=[False, False, False])
        .groupby(["mode", "q_candles", "strategy"], as_index=False)
        .head(1)
    )

selected_valid_df = pd.concat(selected_rows, ignore_index=True)
display(selected_valid_df.sort_values(["mode", "q_candles", "selection_score"], ascending=[True, True, False]))

,period,q_candles,strategy,threshold,trades,total_return,winrate,profit_factor,max_drawdown,avg_trade,mode,min_trades,selection_score
12,valid,2000,TradeSkip + LSTM,0.50,132,-0.00035,0.492424,0.993497,-0.008454,-2.651515e-06,active,100,-1.00035
14,valid,2000,TradeSkip + GRU,0.51,107,-0.00370,0.476636,0.918304,-0.009576,-3.457944e-05,active,100,-1.00370
13,valid,5000,TradeSkip + GRU,0.52,183,-0.00297,0.464481,0.960027,-0.013031,-1.622951e-05,active,100,-1.00297
16,valid,5000,TradeSkip + LSTM,0.51,286,-0.01408,0.454545,0.885398,-0.024534,-4.923077e-05,active,100,-1.01408
15,valid,10000,TradeSkip + GRU,0.53,114,-0.00858,0.421053,0.823093,-0.012019,-7.526316e-05,active,100,-1.00858
17,valid,10000,TradeSkip + LSTM,0.54,142,-0.01434,0.422535,0.780129,-0.020829,-1.009859e-04,active,100,-1.01434
6,valid,2000,TradeSkip + GRU,0.52,74,0.00019,0.486486,1.006340,-0.007022,2.567568e-06,balanced,50,0.00019
7,valid,2000,TradeSkip + LSTM,0.52,93,0.00002,0.483871,1.000522,-0.006624,2.150538e-07,balanced,50,0.00002
8,valid,5000,TradeSkip + GRU,0.52,183,-0.00297,0.464481,0.960027,-0.013031,-1.622951e-05,balanced,50,-1.00297
9,valid,5000,TradeSkip + LSTM,0.54,65,-0.00762,0.415385,0.749589,-0.016144,-1.172308e-04,balanced,50,-1.00762


## Test выбранных режимов

Теперь переносим выбранные thresholds на test и сравниваем с rule baseline.

In [7]:
test_rows = []
for _, row in selected_valid_df.iterrows():
    model_type = row["strategy"].split("+")[-1].strip().lower()
    raw = raw_by_model[model_type]
    test_signals = select_period(raw, TEST_START_TS, None, q_candles=int(row["q_candles"]))
    item = summarize_trades(row["strategy"], test_signals, float(row["threshold"]), int(row["q_candles"]), "test")
    item["mode"] = row["mode"]
    item["valid_threshold"] = row["threshold"]
    item["valid_trades"] = row["trades"]
    item["valid_return"] = row["total_return"]
    item["valid_pf"] = row["profit_factor"]
    test_rows.append(item)

for q_candles in Q_CANDLES_LIST:
    rule_signals = generate_rule_based_signal_history(price_df, max_rows=q_candles)
    item = summarize_trades("Rule baseline", rule_signals, 0.0, q_candles, "test")
    item["mode"] = "baseline"
    item["valid_threshold"] = np.nan
    item["valid_trades"] = np.nan
    item["valid_return"] = np.nan
    item["valid_pf"] = np.nan
    test_rows.append(item)

test_df = pd.DataFrame(test_rows).sort_values(["q_candles", "total_return", "profit_factor"], ascending=[True, False, False])
display(test_df)

,period,q_candles,strategy,threshold,trades,total_return,winrate,profit_factor,max_drawdown,avg_trade,mode,valid_threshold,valid_trades,valid_return,valid_pf
18,test,2000,Rule baseline,0.00,163,0.01640,0.558282,1.259864,-0.009771,0.000101,baseline,NaN,NaN,NaN,NaN
14,test,2000,TradeSkip + GRU,0.51,95,0.01414,0.578947,1.397862,-0.006504,0.000149,active,0.51,107.0,-0.00370,0.918304
0,test,2000,TradeSkip + GRU,0.52,71,0.01049,0.577465,1.400076,-0.003687,0.000148,quality,0.52,74.0,0.00019,1.006340
6,test,2000,TradeSkip + GRU,0.52,71,0.01049,0.577465,1.400076,-0.003687,0.000148,balanced,0.52,74.0,0.00019,1.006340
12,test,2000,TradeSkip + LSTM,0.50,108,0.00823,0.527778,1.188588,-0.010802,0.000076,active,0.50,132.0,-0.00035,0.993497
1,test,2000,TradeSkip + LSTM,0.52,63,0.00725,0.555556,1.292575,-0.007553,0.000115,quality,0.52,93.0,0.00002,1.000522
7,test,2000,TradeSkip + LSTM,0.52,63,0.00725,0.555556,1.292575,-0.007553,0.000115,balanced,0.52,93.0,0.00002,1.000522
19,test,5000,Rule baseline,0.00,420,0.00685,0.492857,1.039735,-0.023458,0.000016,baseline,NaN,NaN,NaN,NaN
16,test,5000,TradeSkip + LSTM,0.51,250,0.00610,0.500000,1.058429,-0.018680,0.000024,active,0.51,286.0,-0.01408,0.885398
2,test,5000,TradeSkip + GRU,0.52,187,0.00429,0.502674,1.055134,-0.011369,0.000023,quality,0.52,183.0,-0.00297,0.960027


## Короткая сводка

Если TradeSkip не обгоняет baseline по total_return, но выигрывает по PF/winrate/drawdown, это можно описывать как AI-фильтр качества входов.

In [8]:
summary = test_df.copy()
summary["return_per_trade"] = np.where(summary["trades"] > 0, summary["total_return"] / summary["trades"], 0)
summary = summary.sort_values(["q_candles", "profit_factor", "winrate"], ascending=[True, False, False])
display(summary[["q_candles", "mode", "strategy", "threshold", "trades", "total_return", "return_per_trade", "winrate", "profit_factor", "max_drawdown"]])

,q_candles,mode,strategy,threshold,trades,total_return,return_per_trade,winrate,profit_factor,max_drawdown
0,2000,quality,TradeSkip + GRU,0.52,71,0.01049,0.000148,0.577465,1.400076,-0.003687
6,2000,balanced,TradeSkip + GRU,0.52,71,0.01049,0.000148,0.577465,1.400076,-0.003687
14,2000,active,TradeSkip + GRU,0.51,95,0.01414,0.000149,0.578947,1.397862,-0.006504
1,2000,quality,TradeSkip + LSTM,0.52,63,0.00725,0.000115,0.555556,1.292575,-0.007553
7,2000,balanced,TradeSkip + LSTM,0.52,63,0.00725,0.000115,0.555556,1.292575,-0.007553
18,2000,baseline,Rule baseline,0.00,163,0.01640,0.000101,0.558282,1.259864,-0.009771
12,2000,active,TradeSkip + LSTM,0.50,108,0.00823,0.000076,0.527778,1.188588,-0.010802
16,5000,active,TradeSkip + LSTM,0.51,250,0.00610,0.000024,0.500000,1.058429,-0.018680
2,5000,quality,TradeSkip + GRU,0.52,187,0.00429,0.000023,0.502674,1.055134,-0.011369
8,5000,balanced,TradeSkip + GRU,0.52,187,0.00429,0.000023,0.502674,1.055134,-0.011369
